# LJP Criminal Dataset - Reason 컬럼 전처리 데모

이 노트북은 `lbox/lbox_open` 데이터셋의 `ljp_criminal` 서브셋에서 `reason` 컬럼을 전처리하는 방법을 보여줍니다.

**핵심 원칙: 원본 텍스트 보존**
- 법률 문서의 모든 내용을 유지
- 최소한의 공백 정규화만 수행
- 리스트 마커, 법률 조항, 특수문자 모두 보존

## 1. 환경 설정

In [ ]:
# 필요한 라이브러리 설치
!pip install datasets huggingface-hub -q

In [ ]:
from datasets import load_dataset
import re
from typing import Optional

## 2. 데이터셋 로드

In [ ]:
# ljp_criminal 데이터셋 로드
dataset = load_dataset("lbox/lbox_open", "ljp_criminal", split="train")

print(f"총 샘플 수: {len(dataset)}")
print(f"컬럼: {dataset.column_names}")

## 3. 전처리 함수 정의

In [ ]:
def preprocess_reason(text: str) -> str:
    """
    reason 컬럼 전처리 함수
    
    원본 텍스트를 최대한 유지하면서 다음만 수행:
    1. 줄 끝 공백 제거
    2. 리스트 마커 뒤 공백 정규화 (단일 공백)
    3. 숫자 섹션 헤더 뒤 공백 정규화 (단일 공백)
    4. 연속된 빈 줄 정리 (최대 2개)
    5. 전체 문서 앞뒤 공백 제거
    
    Args:
        text: 원본 reason 텍스트
        
    Returns:
        전처리된 reason 텍스트
    """
    if not text or not isinstance(text, str):
        return text
    
    # 1. 줄 단위로 분리
    lines = text.split('\n')
    
    # 2. 각 줄 처리
    processed_lines = []
    for line in lines:
        # 줄 끝 공백 제거
        line = line.rstrip()
        
        # 리스트 마커로 시작하는 경우
        if re.match(r'^\s*[○●•\-\*]\s+', line):
            line = line.lstrip()
            line = re.sub(r'^([○●•\-\*])\s+', r'\1 ', line)
        
        # 숫자 섹션 헤더인 경우
        elif re.match(r'^\s*\d+\.\s+', line):
            line = line.lstrip()
            line = re.sub(r'^(\d+\.)\s+', r'\1 ', line)
        
        # 일반 헤더로 보이는 경우
        elif line.strip() and len(line.strip()) < 50:
            if re.search(r'(이유|의무|명령|면제)$', line.strip()):
                line = line.strip()
        
        processed_lines.append(line)
    
    # 3. 다시 합치기
    result = '\n'.join(processed_lines)
    
    # 4. 연속된 빈 줄 정리
    result = re.sub(r'\n{3,}', '\n\n', result)
    
    # 5. 전체 앞뒤 공백 제거
    result = result.strip()
    
    return result

In [ ]:
def validate_preprocessing(original: str, preprocessed: str) -> dict:
    """
    전처리 검증 함수
    """
    # 공백을 모두 제거하고 비교
    orig_no_space = re.sub(r'\s+', '', original)
    prep_no_space = re.sub(r'\s+', '', preprocessed)
    
    # 주요 패턴 개수 확인
    patterns_to_check = {
        '리스트_마커_○': r'○',
        '리스트_마커_-': r'^-\s',
        '법률_조항': r'제\d+조',
        '괄호': r'\([^)]+\)',
    }
    
    pattern_counts = {}
    for name, pattern in patterns_to_check.items():
        orig_count = len(re.findall(pattern, original, re.MULTILINE))
        prep_count = len(re.findall(pattern, preprocessed, re.MULTILINE))
        pattern_counts[name] = {
            'original': orig_count,
            'preprocessed': prep_count,
            'match': orig_count == prep_count
        }
    
    return {
        'content_preserved': orig_no_space == prep_no_space,
        'original_length': len(original),
        'preprocessed_length': len(preprocessed),
        'length_diff': len(preprocessed) - len(original),
        'pattern_counts': pattern_counts,
        'all_patterns_match': all(v['match'] for v in pattern_counts.values())
    }

## 4. 단일 샘플 전처리 예시

In [ ]:
# Row 83 샘플 확인 (원본)
sample_idx = 83
original_text = dataset[sample_idx]['reason']

print("=== 원본 텍스트 ===")
print(original_text)
print(f"\n길이: {len(original_text)} 문자")

In [ ]:
# 전처리 수행
preprocessed_text = preprocess_reason(original_text)

print("=== 전처리된 텍스트 ===")
print(preprocessed_text)
print(f"\n길이: {len(preprocessed_text)} 문자")

In [ ]:
# 전처리 검증
validation = validate_preprocessing(original_text, preprocessed_text)

print("=== 검증 결과 ===")
print(f"✅ 내용 보존: {validation['content_preserved']}")
print(f"📊 길이 변화: {validation['original_length']} → {validation['preprocessed_length']} ({validation['length_diff']:+d})")
print(f"✅ 패턴 일치: {validation['all_patterns_match']}")
print("\n패턴 개수:")
for pattern_name, counts in validation['pattern_counts'].items():
    if counts['original'] > 0 or counts['preprocessed'] > 0:
        match_symbol = '✓' if counts['match'] else '✗'
        print(f"  {pattern_name}: {counts['original']} → {counts['preprocessed']} ({match_symbol})")

## 5. 전체 데이터셋 전처리

In [ ]:
def preprocess_dataset(dataset, validate=True):
    """
    데이터셋 전체 전처리
    """
    def process_example(example):
        original = example['reason']
        preprocessed = preprocess_reason(original)
        
        # 원본도 유지 (비교를 위해)
        example['reason_original'] = original
        example['reason_preprocessed'] = preprocessed
        
        if validate:
            validation = validate_preprocessing(original, preprocessed)
            example['preprocessing_valid'] = validation['content_preserved']
            example['preprocessing_length_diff'] = validation['length_diff']
        
        return example
    
    return dataset.map(process_example)

In [ ]:
# 전체 데이터셋 전처리 (시간이 걸릴 수 있습니다)
print("전체 데이터셋 전처리 시작...")
preprocessed_dataset = preprocess_dataset(dataset)
print("전처리 완료!")

print(f"\n새로운 컬럼: {preprocessed_dataset.column_names}")

## 6. 전처리 품질 검증

In [ ]:
# 모든 샘플이 올바르게 전처리되었는지 확인
valid_count = sum(preprocessed_dataset['preprocessing_valid'])
total_count = len(preprocessed_dataset)

print(f"전처리 성공률: {valid_count}/{total_count} ({valid_count/total_count*100:.2f}%)")

# 길이 변화 통계
length_diffs = preprocessed_dataset['preprocessing_length_diff']
print(f"\n길이 변화 통계:")
print(f"  평균: {sum(length_diffs)/len(length_diffs):.2f} 문자")
print(f"  최소: {min(length_diffs)} 문자")
print(f"  최대: {max(length_diffs)} 문자")

# 제거된 불필요한 공백 총합
total_removed = sum(abs(d) for d in length_diffs if d < 0)
print(f"\n제거된 불필요한 공백 총합: {total_removed} 문자")

## 7. 여러 샘플 비교

In [ ]:
# 랜덤 샘플 몇 개 비교
import random

sample_indices = random.sample(range(len(preprocessed_dataset)), 5)

for idx in sample_indices:
    sample = preprocessed_dataset[idx]
    print(f"\n{'='*80}")
    print(f"샘플 {idx}")
    print('='*80)
    print(f"원본 길이: {len(sample['reason_original'])}")
    print(f"전처리 후: {len(sample['reason_preprocessed'])}")
    print(f"차이: {sample['preprocessing_length_diff']}")
    print(f"유효: {'✅' if sample['preprocessing_valid'] else '❌'}")
    print(f"\n전처리 결과 (처음 200자):")
    print(sample['reason_preprocessed'][:200])
    print("...")

## 8. 데이터셋 저장 (선택사항)

In [ ]:
# 전처리된 데이터셋을 로컬에 저장
# preprocessed_dataset.save_to_disk('./ljp_criminal_preprocessed')
# print("데이터셋이 './ljp_criminal_preprocessed'에 저장되었습니다.")

# CSV로 저장 (일부 컬럼만)
# import pandas as pd
# df = pd.DataFrame({
#     'id': preprocessed_dataset['id'],
#     'reason_preprocessed': preprocessed_dataset['reason_preprocessed'],
#     'label': preprocessed_dataset['label']
# })
# df.to_csv('ljp_criminal_reason_preprocessed.csv', index=False)
# print("CSV 파일이 저장되었습니다.")

## 요약

이 전처리 파이프라인은:

✅ **원본 보존**: 모든 법률 텍스트의 내용을 완전히 유지

✅ **최소 정규화**: 불필요한 공백만 제거

✅ **구조 유지**: 리스트 마커, 섹션 헤더, 법률 조항 모두 보존

✅ **검증 가능**: 각 전처리 단계를 검증하여 무결성 보장

이 방식을 사용하면 법률 문서의 중요한 정보를 손실하지 않으면서도 모델 학습에 적합한 형태로 데이터를 준비할 수 있습니다.